# 🤖 Customer Segmentation — Complete ML Pipeline
## What This Notebook Covers
The ENTIRE ML pipeline from raw data to final segment assignments.
Every line of code is commented. Every concept is explained.
Designed for learners and presenters.

## The Big Picture
```
Raw Customer Data
     ↓
StandardScaler (normalize all features to same range)
     ↓
Find Optimal K (Elbow + Silhouette)
     ↓
K-Means Clustering (assign each customer to 1 of K groups)
     ↓
PCA (reduce 13 dimensions → 2 for visualization)
     ↓
Segment Profiling (name each group by its behavior)
     ↓
Export for Marketing Action
```

## Three ML Algorithms
| Algorithm | Purpose |
|---|---|
| StandardScaler | Normalize features so no single feature dominates |
| K-Means | Assign customers to clusters by minimizing within-cluster variance |
| PCA | Reduce dimensions from 13 to 2 for human-readable visualization |

## 📦 Section 1: Import All Libraries

In [ ]:
# Data manipulation — the core library for working with tabular data
import pandas as pd

# Mathematical operations on arrays — fast, vectorized computations
import numpy as np

# ── Scikit-learn: our complete ML toolkit ─────────────────────────
# K-Means: the primary clustering algorithm
from sklearn.cluster import KMeans

# DBSCAN: density-based clustering — finds noise/outliers too
from sklearn.cluster import DBSCAN

# Agglomerative: hierarchical clustering — builds a tree of clusters
from sklearn.cluster import AgglomerativeClustering

# PCA: Principal Component Analysis — reduces many dimensions to few
from sklearn.decomposition import PCA

# StandardScaler: normalizes features to mean=0, std=1
from sklearn.preprocessing import StandardScaler

# silhouette_score: measures how well-separated clusters are (−1 to 1)
# silhouette_samples: per-sample scores for detailed analysis
from sklearn.metrics import silhouette_score, silhouette_samples

# davies_bouldin_score: alternative clustering quality metric
# Lower is better (0 = perfect separation)
from sklearn.metrics import davies_bouldin_score

# Model serialization — save trained models to disk
import joblib

# OS operations — create directories, check file paths
import os

# Visualization
import matplotlib.pyplot as plt
import matplotlib.cm as cm       # Color maps for silhouette plots
import seaborn as sns
from scipy.cluster.hierarchy import dendrogram, linkage  # For hierarchical viz

import warnings
warnings.filterwarnings("ignore")

# Fix random seed so results are reproducible — same result every run
np.random.seed(42)

print("All libraries imported ✅")

## 📂 Section 2: Load Data and Define Features

In [ ]:
# Load the customer dataset generated by generate_data.py
df = pd.read_csv("../data/customers.csv")

# Define which columns to use for clustering
# These 13 features capture the full behavioral profile of each customer
CLUSTER_FEATURES = [
    'annual_income',        # Customer's yearly income — proxy for budget
    'spending_score',       # Score 1-100 indicating spending willingness
    'recency_days',         # Days since last purchase — lower = more active
    'frequency',            # Purchases per year — higher = more engaged
    'monetary',             # Total annual spend — the $$ value
    'avg_order_value',      # Spend per transaction — basket size
    'online_purchase_ratio',# How digitally active they are (0-1)
    'loyalty_years',        # Years as customer — relationship depth
    'discount_usage_rate',  # How price-sensitive they are (0-1)
    'returns_rate',         # Returns as proxy for satisfaction (0-1)
    'support_tickets',      # Service interactions — friction measure
    'clv_score',            # Customer lifetime value proxy score
    'engagement_score',     # Composite engagement metric
]

# Extract only the clustering features into matrix X
# We do NOT include customer_id, archetype, gender — non-numeric/leakage
X = df[CLUSTER_FEATURES].copy()

print(f"Feature matrix shape: {X.shape}  ({X.shape[0]} customers × {X.shape[1]} features)")
print(f"\nFeature value ranges (before scaling):")
print(X.describe().loc[['min','max','mean']].round(2))

## 📏 Section 3: StandardScaler — Why Scaling is Critical for K-Means
K-Means uses EUCLIDEAN DISTANCE to measure similarity between customers.
If annual_income ranges from $15,000 to $200,000 (range: 185,000)
and spending_score ranges from 1 to 100 (range: 99),
the income feature would dominate every distance calculation —
not because it's more important, but because its numbers are larger.

StandardScaler transforms every feature to have:
  - Mean = 0 (centered)
  - Standard Deviation = 1 (same spread)
After scaling, all features contribute equally to distance.

In [ ]:
# Initialize the scaler — this object will learn the mean and std
# of each feature from our data
scaler = StandardScaler()

# fit_transform() does two steps in one:
# 1. fit()      — computes mean and std for each of the 13 features
# 2. transform() — applies: scaled_value = (raw_value - mean) / std
# Returns a 2D numpy array with the same shape as X
X_scaled = scaler.fit_transform(X)

# Convert back to DataFrame so we keep column names for readability
X_scaled_df = pd.DataFrame(X_scaled, columns=CLUSTER_FEATURES)

# Demonstrate the effect of scaling with a side-by-side comparison
print("BEFORE scaling:")
print(f"  annual_income — mean: {X['annual_income'].mean():>10,.0f}  "
      f"std: {X['annual_income'].std():>8,.0f}")
print(f"  spending_score — mean: {X['spending_score'].mean():>9.2f}  "
      f"std: {X['spending_score'].std():>8.2f}")

print("\nAFTER scaling (StandardScaler):")
print(f"  annual_income — mean: {X_scaled_df['annual_income'].mean():>10.4f}  "
      f"std: {X_scaled_df['annual_income'].std():>8.4f}")
print(f"  spending_score — mean: {X_scaled_df['spending_score'].mean():>9.4f}  "
      f"std: {X_scaled_df['spending_score'].std():>8.4f}")
print("\nBoth features now have the same scale — fair comparison ✅")

# Save the fitted scaler so we can reuse it in production
# CRITICAL: We must use the SAME scaler when scoring new customers
os.makedirs("../models", exist_ok=True)
joblib.dump(scaler, "../models/scaler.pkl")
print("Scaler saved to ../models/scaler.pkl ✅")

## 🔍 Section 4: Finding Optimal K — Elbow Method
How many clusters should we use? We test K=2 to K=10.

**Inertia (Within-Cluster Sum of Squares):**
As K increases, clusters get smaller, so inertia always decreases.
But at some point the decrease becomes marginal — that's the "elbow."
Choosing K at the elbow means adding more clusters gives little benefit.

**Silhouette Score:**
Measures how well each point fits its own cluster vs the next-nearest.
Range: -1 (wrong cluster) to +1 (perfect cluster).
Higher silhouette = better separation.

In [ ]:
# Define the range of K values to test
k_range = range(2, 11)

# Lists to store metrics for each K value
inertias           = []   # Within-cluster sum of squares (elbow)
silhouette_scores  = []   # Cluster separation quality
davies_bouldin_scores = [] # Alternative metric (lower = better)

print("Testing K values from 2 to 10...")
print(f"{'K':>3} | {'Inertia':>12} | {'Silhouette':>12} | {'Davies-Bouldin':>15}")
print("-" * 50)

for k in k_range:
    # Initialize K-Means with k-means++ initialization
    # k-means++ spreads initial centroids intelligently — avoids bad starts
    km = KMeans(
        n_clusters=k,    # Number of clusters to find
        init='k-means++',# Smart initialization (not random)
        n_init=20,       # Run 20 times with different seeds, keep best
        max_iter=500,    # Max iterations per run
        random_state=42  # Fixed seed for reproducibility
    )

    # fit_predict() trains and returns cluster labels in one step
    labels = km.fit_predict(X_scaled)

    # inertia_ is the sum of squared distances of each point to its centroid
    # Lower = tighter clusters
    inertias.append(km.inertia_)

    # silhouette_score computes the mean silhouette across all samples
    # Requires at least 2 clusters and more samples than clusters
    silhouette_scores.append(silhouette_score(X_scaled, labels))

    # davies_bouldin_score: ratio of within-cluster to between-cluster spread
    # Lower is better — 0 is perfect
    davies_bouldin_scores.append(davies_bouldin_score(X_scaled, labels))

    print(f"{k:>3} | {km.inertia_:>12,.0f} | "
          f"{silhouette_scores[-1]:>12.4f} | "
          f"{davies_bouldin_scores[-1]:>15.4f}")

## 📊 Section 5: Visualize Elbow + Silhouette to Choose K

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- Plot 1: Elbow Curve (Inertia) ---
axes[0].plot(list(k_range), inertias, 'bo-', linewidth=2, markersize=7)
# Mark the optimal K with a vertical dashed line
axes[0].axvline(5, color='red', linestyle='--', linewidth=1.5, label='K=5 (optimal)')
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Inertia (WCSS)')
axes[0].set_title('Elbow Method', fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)
# Annotate: the "elbow" is where the curve bends most sharply
# After that point, adding clusters doesn't reduce inertia much

# --- Plot 2: Silhouette Score ---
axes[1].plot(list(k_range), silhouette_scores, 'gs-', linewidth=2, markersize=7)
axes[1].axvline(5, color='red', linestyle='--', linewidth=1.5, label='K=5')
# Mark the maximum silhouette score with a star
best_k_idx = np.argmax(silhouette_scores)
axes[1].scatter(list(k_range)[best_k_idx], silhouette_scores[best_k_idx],
                s=150, zorder=5, color='gold', marker='*',
                label=f'Best: K={list(k_range)[best_k_idx]}')
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Silhouette Score (higher = better)')
axes[1].set_title('Silhouette Score', fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

# --- Plot 3: Davies-Bouldin Score ---
axes[2].plot(list(k_range), davies_bouldin_scores, 'r^-', linewidth=2, markersize=7)
axes[2].axvline(5, color='red', linestyle='--', linewidth=1.5, label='K=5')
# Mark the minimum (lower is better for DB score)
best_db_idx = np.argmin(davies_bouldin_scores)
axes[2].scatter(list(k_range)[best_db_idx], davies_bouldin_scores[best_db_idx],
                s=150, zorder=5, color='gold', marker='*',
                label=f'Best: K={list(k_range)[best_db_idx]}')
axes[2].set_xlabel('Number of Clusters (K)')
axes[2].set_ylabel('Davies-Bouldin Score (lower = better)')
axes[2].set_title('Davies-Bouldin Score', fontweight='bold')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.suptitle('Finding Optimal K — Three Methods', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nRecommended K by Silhouette:     {list(k_range)[np.argmax(silhouette_scores)]}")
print(f"Recommended K by Davies-Bouldin: {list(k_range)[np.argmin(davies_bouldin_scores)]}")
print(f"Domain knowledge (5 archetypes): K=5")
print("→ Selecting K=5 ✅")

## 🎯 Section 6: Train Final K-Means Model (K=5)
K-Means Algorithm — How it Works:
1. **Initialize**: Place K=5 centroids using K-Means++ (smart spread)
2. **Assign**: Each customer goes to the nearest centroid
3. **Update**: Move each centroid to the mean of its assigned customers
4. **Repeat**: Steps 2-3 until centroids stop moving (convergence)

We run this 50 times with different random starts and keep the
best result (lowest inertia) — this avoids getting stuck in
bad local optima.

In [ ]:
OPTIMAL_K = 5  # Chosen from elbow + silhouette + domain knowledge

# Train the final K-Means model with optimized settings
kmeans = KMeans(
    n_clusters=OPTIMAL_K,  # We want exactly 5 clusters
    init='k-means++',      # Spreads initial centroids intelligently
    n_init=50,             # Run 50 independent times — keep the best
    max_iter=1000,         # Maximum iterations per run (convergence)
    tol=1e-6,              # Stop when centroids move less than this
    random_state=42        # Fixed seed so results are the same every run
)

print(f"Training K-Means with K={OPTIMAL_K}...")
print(f"Running {kmeans.n_init} initializations — this ensures we find")
print("the global optimum rather than a local one.\n")

# fit_predict() trains the model AND assigns labels in one step
# Returns array of integers 0..K-1, one per customer
cluster_labels = kmeans.fit_predict(X_scaled)

# Measure quality of the final clustering
final_silhouette = silhouette_score(X_scaled, cluster_labels)
final_db         = davies_bouldin_score(X_scaled, cluster_labels)

print(f"Final model trained ✅")
print(f"  Inertia:              {kmeans.inertia_:,.0f}")
print(f"  Silhouette Score:     {final_silhouette:.4f}  (higher = better)")
print(f"  Davies-Bouldin:       {final_db:.4f}  (lower = better)")
print(f"  Iterations to converge: {kmeans.n_iter_}")
print(f"\nCluster sizes:")
# Count how many customers ended up in each cluster
unique, counts = np.unique(cluster_labels, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  Cluster {u}: {c:,} customers ({c/len(cluster_labels)*100:.1f}%)")

## 🔬 Section 7: PCA — Visualize Clusters in 2D
We have 13 features — impossible to visualize directly.
PCA finds the 2 linear combinations of our 13 features that
capture the most variance in the data.

Think of it as: "If I had to project a 3D sculpture onto a 2D
photo, what angle gives the most informative shadow?" PCA finds
the mathematically optimal angle for any number of dimensions.

PC1 (x-axis): The single direction that explains the most variation
PC2 (y-axis): The direction perpendicular to PC1 with next-most variation

In [ ]:
# Initialize PCA with 2 components — reduces 13D to 2D
# random_state ensures reproducibility (PCA uses randomized SVD internally)
pca = PCA(n_components=2, random_state=42)

# fit_transform():
# 1. fit() — learns the principal components from scaled data
# 2. transform() — projects each customer onto the 2 PCA axes
X_pca = pca.fit_transform(X_scaled)

# explained_variance_ratio_ tells us how much of the original
# data's variance is captured by each component
var1 = pca.explained_variance_ratio_[0] * 100  # PC1 variance %
var2 = pca.explained_variance_ratio_[1] * 100  # PC2 variance %
total_var = var1 + var2                          # Combined

print(f"PCA Results:")
print(f"  PC1 explains: {var1:.1f}% of total variance")
print(f"  PC2 explains: {var2:.1f}% of total variance")
print(f"  Combined:     {total_var:.1f}% of total variance captured in 2D")
print(f"\nThe 2D plot preserves {total_var:.1f}% of information from 13 features")
print(f"(We lose {100-total_var:.1f}% — acceptable for visualization purposes)")

# Add PCA coordinates to the main DataFrame
df['pca_x']     = X_pca[:, 0]  # PC1 coordinate for each customer
df['pca_y']     = X_pca[:, 1]  # PC2 coordinate for each customer
df['cluster']   = cluster_labels  # Which cluster was assigned

## 🗺️ Section 8: Visualize Clusters on PCA Scatter Plot

In [ ]:
# Define colors for each cluster (0 through 4)
cluster_colors = ['#F39C12', '#3498DB', '#2ECC71', '#E74C3C', '#9B59B6']

# Temporary names — we'll assign proper business names in next section
temp_names = ['Cluster 0', 'Cluster 1', 'Cluster 2', 'Cluster 3', 'Cluster 4']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Plot 1: K-Means clusters on PCA axes ---
for cluster_id, (color, name) in enumerate(zip(cluster_colors, temp_names)):
    # Select all points belonging to this cluster
    mask = df['cluster'] == cluster_id
    # Plot each cluster's customers as a separate scatter layer
    # alpha=0.5 makes overlapping points visible
    axes[0].scatter(df.loc[mask, 'pca_x'], df.loc[mask, 'pca_y'],
                    c=color, label=name, alpha=0.5, s=15, edgecolors='none')

# Plot the cluster centroids — these are the "center of mass" of each cluster
# pca.transform() projects the 13D centroids into 2D PCA space
centroids_pca = pca.transform(kmeans.cluster_centers_)
axes[0].scatter(centroids_pca[:, 0], centroids_pca[:, 1],
                c='white', s=200, marker='*', zorder=5,
                edgecolors='black', linewidth=1.5, label='Centroids')
axes[0].set_xlabel(f'PC1 ({var1:.1f}% variance explained)')
axes[0].set_ylabel(f'PC2 ({var2:.1f}% variance explained)')
axes[0].set_title('K-Means Clusters (K=5) on PCA Axes', fontweight='bold')
axes[0].legend(fontsize=8, markerscale=2)

# --- Plot 2: True archetypes on same PCA axes (ground truth validation) ---
archetype_colors_map = {'A':'#F39C12','B':'#3498DB','C':'#2ECC71',
                         'D':'#E74C3C','E':'#9B59B6'}
archetype_names = {'A':'Premium Loyalists','B':'Occasional Shoppers',
                   'C':'Bargain Hunters','D':'At-Risk High-Value',
                   'E':'Young Explorers'}
for arch, color in archetype_colors_map.items():
    mask = df['archetype'] == arch
    axes[1].scatter(df.loc[mask, 'pca_x'], df.loc[mask, 'pca_y'],
                    c=color, label=archetype_names[arch],
                    alpha=0.5, s=15, edgecolors='none')
axes[1].set_xlabel(f'PC1 ({var1:.1f}% variance explained)')
axes[1].set_ylabel(f'PC2 ({var2:.1f}% variance explained)')
axes[1].set_title('True Archetypes on PCA Axes (Ground Truth)', fontweight='bold')
axes[1].legend(fontsize=8, markerscale=2)

plt.suptitle(f'K-Means vs True Archetypes — PCA Visualization\n'
             f'({total_var:.1f}% variance preserved)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print("Visual similarity between plots = K-Means recovered the true groups ✅")

## 📊 Section 9: Silhouette Analysis — Per-Customer Scores
Silhouette score for each customer = how similar they are to their
own cluster vs the next-nearest cluster.
  - Score near +1: customer clearly belongs in this cluster
  - Score near 0: customer is on the boundary between two clusters
  - Score near -1: customer is likely in the wrong cluster

In [ ]:
# Compute individual silhouette score for every single customer
# Returns an array of length n_customers
sample_silhouettes = silhouette_samples(X_scaled, cluster_labels)

# Compute the mean score for reference
mean_sil = sample_silhouettes.mean()

fig, ax = plt.subplots(figsize=(10, 7))

y_lower = 10  # Vertical position for the first cluster's silhouettes

for cluster_id in range(OPTIMAL_K):
    # Extract silhouette scores for this cluster
    cluster_sil = sample_silhouettes[cluster_labels == cluster_id]

    # Sort scores so the plot looks clean (ascending order)
    cluster_sil.sort()

    # Size of this cluster determines its height in the plot
    cluster_size = cluster_sil.shape[0]
    y_upper = y_lower + cluster_size

    # Fill horizontal bars — each bar is one customer's silhouette
    color = cm.nipy_spectral(float(cluster_id) / OPTIMAL_K)
    ax.fill_betweenx(np.arange(y_lower, y_upper),
                     0, cluster_sil,
                     facecolor=color, edgecolor=color, alpha=0.7)

    # Label the cluster at its vertical center
    ax.text(-0.05, y_lower + 0.5 * cluster_size, f'Cluster {cluster_id}')

    # Leave a gap between clusters for visual clarity
    y_lower = y_upper + 10

# Add vertical line at the mean silhouette score
ax.axvline(mean_sil, color='red', linestyle='--', linewidth=2,
           label=f'Mean Silhouette = {mean_sil:.3f}')

ax.set_xlabel('Silhouette Coefficient')
ax.set_ylabel('Customers (sorted within each cluster)')
ax.set_title(f'Silhouette Analysis — K={OPTIMAL_K}\n'
             f'Wider bars = larger cluster, further right = better fit',
             fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Mean silhouette score: {mean_sil:.4f}")
print("Interpretation: >0.5 = good structure, >0.7 = strong structure")

## 🏷️ Section 10: Segment Profiling — Naming the Clusters
K-Means gives us numbered clusters (0, 1, 2, 3, 4).
We need to name them by looking at the mean feature values
of each cluster and matching to business archetypes.

In [ ]:
# Add cluster labels to the original DataFrame
df['cluster'] = cluster_labels

# Compute mean of all CLUSTER_FEATURES grouped by cluster
cluster_profiles = df.groupby('cluster')[CLUSTER_FEATURES].mean()

# Round for readability
print("=== CLUSTER PROFILES (mean feature values) ===\n")
print(cluster_profiles.round(2).T.to_string())

print("\n\n=== KEY DIFFERENTIATING FEATURES ===")
# Print the top 4 most variable features across clusters
# (these are what distinguish the clusters most)
feature_variance = cluster_profiles.std()
top_features = feature_variance.sort_values(ascending=False).head(6)
print("\nFeatures with highest variation across clusters:")
for feat, var in top_features.items():
    print(f"  {feat:<30}: std={var:.3f}")

## 💾 Section 11: Hierarchical Clustering — Dendrogram
Agglomerative Clustering builds a tree (dendrogram) showing
how customers merge into groups. We use a sample of 100 customers
to visualize the hierarchy — the full 2000 would be too dense.

In [ ]:
# Take a stratified sample — 20 customers per cluster
# This ensures all clusters are represented in the dendrogram
sample_indices = []
for c in range(OPTIMAL_K):
    # Get indices of customers in this cluster
    cluster_idx = np.where(cluster_labels == c)[0]
    # Randomly sample 20 (or all if fewer than 20)
    n_sample = min(20, len(cluster_idx))
    sampled = np.random.choice(cluster_idx, n_sample, replace=False)
    sample_indices.extend(sampled)

# Extract scaled features for the sampled customers
X_sample = X_scaled[sample_indices]
labels_sample = cluster_labels[sample_indices]

# linkage() computes hierarchical clustering
# method='ward' minimizes variance within merged clusters
# This is the same objective as K-Means
Z = linkage(X_sample, method='ward')

plt.figure(figsize=(14, 6))
# dendrogram() draws the tree — each leaf is a customer
# color_threshold cuts the tree at a height to show K clusters
dendrogram(Z, color_threshold=0.7*max(Z[:,2]),
           leaf_rotation=90, leaf_font_size=6,
           above_threshold_color='gray')
plt.axhline(y=0.7*max(Z[:,2]), color='red', linestyle='--',
            label='Cut for K=5 clusters', linewidth=1.5)
plt.xlabel('Customer Index (sample of 100)')
plt.ylabel('Linkage Distance (Ward)')
plt.title('Hierarchical Clustering Dendrogram\n'
          '(Cut the tree where you want N clusters)',
          fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

## 📊 Section 12: PCA Deep Dive — What Does Each Component Represent?

In [ ]:
# pca.components_ is a matrix of shape (n_components, n_features)
# Each row is a principal component — a weighted combination of features
# Large positive weight: feature contributes positively to this PC
# Large negative weight: feature contributes negatively
components_df = pd.DataFrame(
    pca.components_,           # Shape: (2, 13)
    columns=CLUSTER_FEATURES,  # Feature names as column headers
    index=['PC1', 'PC2']       # Component names as row index
)

fig, axes = plt.subplots(2, 1, figsize=(13, 8))

for i, pc in enumerate(['PC1', 'PC2']):
    # Get the loadings (weights) for this component
    loadings = components_df.loc[pc]

    # Sort by absolute value to see which features matter most
    sorted_loadings = loadings.reindex(loadings.abs().sort_values().index)

    # Color: positive loadings = blue, negative = red
    colors = ['#3498DB' if v > 0 else '#E74C3C' for v in sorted_loadings]
    axes[i].barh(sorted_loadings.index, sorted_loadings.values,
                 color=colors, edgecolor='white', alpha=0.8)
    axes[i].axvline(0, color='black', linewidth=0.8)
    axes[i].set_xlabel('Loading (weight in linear combination)')
    axes[i].set_title(f'{pc} — Feature Loadings '
                      f'(explains {pca.explained_variance_ratio_[i]*100:.1f}% variance)',
                      fontweight='bold')
    axes[i].grid(axis='x', alpha=0.3)

plt.suptitle('PCA Component Loadings — What Each Axis Represents',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("PC1 — features with large positive loading:")
print(components_df.loc['PC1'].sort_values(ascending=False).head(4))
print("\nPC2 — features with large positive loading:")
print(components_df.loc['PC2'].sort_values(ascending=False).head(4))

## 💾 Section 13: Save All Models and Final Dataset

In [ ]:
# Create models directory if it does not already exist
os.makedirs("../models", exist_ok=True)

# Save the trained K-Means model
# joblib.dump() serializes the Python object to a binary file
joblib.dump(kmeans, "../models/kmeans_model.pkl")
print(f"K-Means saved: {os.path.getsize('../models/kmeans_model.pkl')//1024} KB")

# Save the PCA model — we need this to project new customers into 2D
joblib.dump(pca, "../models/pca_model.pkl")
print(f"PCA saved:     {os.path.getsize('../models/pca_model.pkl')//1024} KB")

# Scaler was already saved in Section 3
print(f"Scaler:        already saved ✅")

# Save the segmented dataset with all labels and PCA coordinates
df['cluster']      = cluster_labels
df['pca_x']        = X_pca[:, 0]
df['pca_y']        = X_pca[:, 1]
os.makedirs("../data", exist_ok=True)
df.to_csv("../data/customers_segmented.csv", index=False)
print(f"\nSegmented dataset saved: {len(df):,} customers with cluster labels")
print(f"Columns: {list(df.columns)}")
print("\nAll models saved ✅ — ready to run: streamlit run app.py")

## 🎯 Section 14: End-to-End New Customer Demo
Test the complete pipeline on a brand-new customer record —
exactly what happens in the Streamlit app when a new customer
is uploaded.

In [ ]:
# Simulate a new customer who was NOT in the training data
new_customer = {
    'annual_income':          85000,  # High earner
    'spending_score':         88,     # Very willing to spend
    'recency_days':           12,     # Bought very recently
    'frequency':              42,     # Buys frequently
    'monetary':               5200,   # High annual spend
    'avg_order_value':        5200/42,# Basket size
    'online_purchase_ratio':  0.3,    # Mostly in-store
    'loyalty_years':          8,      # Long-term customer
    'discount_usage_rate':    0.05,   # Rarely uses discounts
    'returns_rate':           0.02,   # Almost never returns
    'support_tickets':        1,      # Low-friction customer
    'clv_score':              (42*5200)/(12+1), # CLV proxy formula
    'engagement_score':       (1/12*100)+(42*2)+(0.3*20)-(1*3), # engagement
}

# Step 1: Convert to DataFrame — sklearn expects 2D input
X_new = pd.DataFrame([new_customer], columns=CLUSTER_FEATURES)

# Step 2: Scale using the SAME scaler from training
# IMPORTANT: .transform() not .fit_transform() — we must not refit!
# Refitting would compute different mean/std and corrupt the prediction
X_new_scaled = scaler.transform(X_new)

# Step 3: Predict which cluster this customer belongs to
# .predict() uses the trained centroids and assigns to nearest one
predicted_cluster = kmeans.predict(X_new_scaled)[0]

# Step 4: Project to 2D using the SAME PCA from training
X_new_pca = pca.transform(X_new_scaled)

# Map cluster number to business segment name
SEGMENT_NAMES = {
    0: "Premium Loyalists",
    1: "Occasional Shoppers",
    2: "Bargain Hunters",
    3: "At-Risk High-Value",
    4: "Young Explorers"
}

# Step 5: Find distance to all centroids (confidence measure)
# Closer to centroid = more confident assignment
from sklearn.metrics import pairwise_distances_argmin_min
_, distances = pairwise_distances_argmin_min(
    X_new_scaled, kmeans.cluster_centers_)
confidence = 1 - (distances[0] / distances.max())  # Normalized 0-1

print("=" * 55)
print("  NEW CUSTOMER SEGMENT ASSIGNMENT")
print("=" * 55)
print(f"  Predicted Cluster:    {predicted_cluster}")
print(f"  Segment Name:         {SEGMENT_NAMES[predicted_cluster]}")
print(f"  PCA Position:         ({X_new_pca[0,0]:.2f}, {X_new_pca[0,1]:.2f})")
print(f"  Assignment Confidence:{confidence*100:.1f}%")
print("=" * 55)
print("  Characteristics match:")
print("  → High income + high spending + low recency = Premium profile")
print("  → Low discount usage + high loyalty = Loyalist behavior")
print("  → Predicted: Premium Loyalists — strategy: VIP program")